In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import json
import wandb
import tensorflow as tf
from google.colab import userdata

BASE = '/content/drive/MyDrive/Ketastasia/data'

data = np.load(f'{BASE}/dataset_seq_33kp_ready.npz')
X_train, y_train, y_train_idx = data['X_train'], data['y_train'], data['y_train_int']
X_val,   y_val,   y_val_idx   = data['X_val'],   data['y_val'],   data['y_val_int']

with open(f'{BASE}/pipeline2A_metadata.json') as f:
    meta = json.load(f)

n_classes  = len(meta['classes'])
seq_len    = meta['sequence_length']
n_features = meta['n_features_per_frame']

print(f"Loaded: X_train {X_train.shape}, y_train (one-hot) {y_train.shape}, y_train_idx (int) {y_train_idx.shape}")

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

Mounted at /content/drive
Loaded: X_train (5579, 15, 72), y_train (one-hot) (5579, 26), y_train_idx (int) (5579,)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs(f'{BASE}/models', exist_ok=True)

classes = meta['classes']

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

def build_lstm_baseline(hidden=64, n_layers=1, lr=0.001):
    model = Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(hidden, return_sequences=return_seq))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
configs = [
    {'name': 'small',  'hidden': 32,  'n_layers': 1, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'medium', 'hidden': 64,  'n_layers': 1, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'large',  'hidden': 128, 'n_layers': 2, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
]
print(f"მომზადებულია {len(configs)} capacity ვარიანტი (small/medium/large)")

მომზადებულია 3 capacity ვარიანტი (small/medium/large)


In [ ]:
from wandb.integration.keras import WandbMetricsLogger
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

best_val_acc = 0
best_run_name = None
baseline_results = []

for cfg in configs:
    run_name = f"lstm_baseline_{cfg['name']}"

    print(f"\n{'-'*60}\nიწყება Baseline ექსპერიმენტი: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'lstm',
            'dropout_included': False,
            'class_weights_included': False,
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_lstm_baseline(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    # --- gap-ის გამოთვლა: train_acc იმ epoch-ზე, სადაც val_acc საუკეთესო იყო ---
    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Baseline) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    model_path = f'{BASE}/models/{run_name}.keras'
    model.save(model_path)

    baseline_results.append({
        'run_name': run_name, 'hidden': cfg['hidden'], 'n_layers': cfg['n_layers'],
        'train_acc': train_acc_at_best, 'val_acc': val_acc, 'gap': gap,
        'val_loss': val_loss, 'val_f1_macro': report_val['macro avg']['f1-score']
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_run_name = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

baseline_df = pd.DataFrame(baseline_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nBaseline ექსპერიმენტები დასრულდა!\n{'='*60}")
print(baseline_df[['run_name', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])
print(f"\nსაუკეთესო: {best_run_name} (Val Acc: {best_val_acc:.4f})")


------------------------------------------------------------
იწყება Baseline ექსპერიმენტი: lstm_baseline_small
------------------------------------------------------------


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.3716 - loss: 2.1784 - val_accuracy: 0.4192 - val_loss: 1.7916 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6326 - loss: 1.2016 - val_accuracy: 0.5445 - val_loss: 1.4743 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7460 - loss: 0.8521 - val_accuracy: 0.5490 - val_loss: 1.4296 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7998 - loss: 0.6766 - val_accuracy: 0.6016 - val_loss: 1.3942 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8317 - loss: 0.5626 - val_accuracy: 0.5926 - val_loss: 1.4148 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8593 - loss: 0.4700 - val_accuracy: 0.6216 - val_loss: 1.3695 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8812 - loss: 0.4061

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_baseline_small დასრულდა. Val Accuracy: 0.6579, Gap: 0.3163


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5876 - loss: 1.4313 - val_accuracy: 0.5681 - val_loss: 1.5347 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7951 - loss: 0.6485 - val_accuracy: 0.6352 - val_loss: 1.4310 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8579 - loss: 0.4440 - val_accuracy: 0.6252 - val_loss: 1.5234 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8996 - loss: 0.3092 - val_accuracy: 0.6416 - val_loss: 1.6745 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9163 - loss: 0.2525 - val_accuracy: 0.6225 - val_loss: 1.8241 - learning_rate: 0.0010
Epoch 6/40
170/175 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9354 - loss: 0.2031
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9317 - loss: 0.21

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_baseline_large დასრულდა. Val Accuracy: 0.6633, Gap: 0.3363


epoch/accuracy,▁▅▆▆▇▇▇██████████████████████
epoch/epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
epoch/learning_rate,██████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▆▅▆▅▄▇▇▇▇█▇▇▇███████████████
epoch/val_loss,▂▁▂▃▄▅▄▅▅▆▆▇▆▇▇▇▇▇▇██████████
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Baseline ექსპერიმენტები დასრულდა!
               run_name  train_acc   val_acc       gap  val_loss  val_f1_macro
2   lstm_baseline_large   0.999642  0.663339  0.336302  2.362856      0.560203
1  lstm_baseline_medium   0.969349  0.659710  0.309640  1.739257      0.551109
0   lstm_baseline_small   0.974189  0.657895  0.316294  1.740174      0.576692

საუკეთესო: lstm_baseline_large (Val Acc: 0.6633)


In [ ]:
def build_lstm_dropout(dropout_rate, hidden=64, n_layers=1, lr=0.001):
    model = Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(
            hidden,
            return_sequences=return_seq,
            dropout=dropout_rate,
            recurrent_dropout=dropout_rate
        ))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
configs_dropout = [
    {'name': 'dropout_0.1', 'hidden': 64, 'n_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.2', 'hidden': 64, 'n_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.3', 'hidden': 64, 'n_layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.5', 'hidden': 64, 'n_layers': 1, 'dropout': 0.5, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.8', 'hidden': 64, 'n_layers': 1, 'dropout': 0.8, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
]
print(f"მომზადებულია {len(configs_dropout)} dropout მნიშვნელობა (capacity ფიქსირებული: hidden=64, n_layers=1)")

მომზადებულია 5 dropout მნიშვნელობა (capacity ფიქსირებული: hidden=64, n_layers=1)


In [ ]:
best_val_acc_dropout = 0
best_run_name_dropout = None
dropout_results = []

for cfg in configs_dropout:
    run_name = f"lstm_{cfg['name']}"

    print(f"\n{'-'*60}\nიწყება Dropout ექსპერიმენტი: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'lstm',
            'dropout_included': True,
            'class_weights_included': False,
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_lstm_dropout(
        dropout_rate=cfg['dropout'],
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Dropout) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    model_path = f'{BASE}/models/{run_name}.keras'
    model.save(model_path)

    dropout_results.append({
        'run_name': run_name, 'dropout': cfg['dropout'],
        'train_acc': train_acc_at_best, 'val_acc': val_acc, 'gap': gap,
        'val_loss': val_loss, 'val_f1_macro': report_val['macro avg']['f1-score']
    })

    if val_acc > best_val_acc_dropout:
        best_val_acc_dropout = val_acc
        best_run_name_dropout = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

dropout_df = pd.DataFrame(dropout_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nDropout sweep დასრულდა!\n{'='*60}")
print(dropout_df[['run_name', 'dropout', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])
print(f"\nსაუკეთესო: {best_run_name_dropout} (Val Acc: {best_val_acc_dropout:.4f})")


------------------------------------------------------------
იწყება Dropout ექსპერიმენტი: lstm_dropout_0.1
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 61ms/step - accuracy: 0.4218 - loss: 2.0434 - val_accuracy: 0.5027 - val_loss: 1.6368 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.6532 - loss: 1.1298 - val_accuracy: 0.5581 - val_loss: 1.4257 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.7215 - loss: 0.8742 - val_accuracy: 0.5953 - val_loss: 1.3760 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 18s 49ms/step - accuracy: 0.7697 - loss: 0.7314 - val_accuracy: 0.6180 - val_loss: 1.3606 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 60ms/step - accuracy: 0.7976 - loss: 0.6321 - val_accuracy: 0.6152 - val_loss: 1.4262 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.8197 - loss: 0.5684 - val_accuracy: 0.6425 - val_loss: 1.3733 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.8432 - los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_dropout_0.1 დასრულდა. Val Accuracy: 0.6887, Gap: 0.2261


epoch/accuracy,▁▄▅▆▆▇▇▇▇▇█████████████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▅▅▆▆▇▇▇▇▇▇▇█████████████
epoch/val_loss,█▃▁▁▃▁▃▁▃▃▂▃▃▃▂▁▂▂▂▃▃▃▃▂▂▃▃
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.0905 - loss: 3.2264 - val_accuracy: 0.2686 - val_loss: 2.7648 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 60ms/step - accuracy: 0.1771 - loss: 2.9112 - val_accuracy: 0.2677 - val_loss: 2.5856 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.2151 - loss: 2.7504 - val_accuracy: 0.2777 - val_loss: 2.4950 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.2267 - loss: 2.6368 - val_accuracy: 0.2931 - val_loss: 2.4160 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.2457 - loss: 2.5454 - val_accuracy: 0.3294 - val_loss: 2.3103 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.2656 - loss: 2.4526 - val_accuracy: 0.3203 - val_loss: 2.2670 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.2739 - l

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_dropout_0.8 დასრულდა. Val Accuracy: 0.4419, Gap: -0.0471


epoch/accuracy,▁▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,████████████████████████████▁▁▁▁
epoch/loss,█▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▂▃▃▄▅▄▅▅▆▅▆▆▆▆▆▇▇▇▇▇███▇█████
epoch/val_loss,█▇▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Dropout sweep დასრულდა!
           run_name  dropout  train_acc   val_acc       gap  val_loss  \
0  lstm_dropout_0.1      0.1   0.914859  0.688748  0.226112  1.406332   
2  lstm_dropout_0.3      0.3   0.858218  0.678766  0.179452  1.437246   
1  lstm_dropout_0.2      0.2   0.895501  0.676951  0.218550  1.575322   
3  lstm_dropout_0.5      0.5   0.731852  0.625227  0.106625  1.506420   
4  lstm_dropout_0.8      0.8   0.394874  0.441924 -0.047050  1.883376   

   val_f1_macro  
0      0.576775  
2      0.564893  
1      0.544039  
3      0.503797  
4      0.280766  

საუკეთესო: lstm_dropout_0.1 (Val Acc: 0.6887)


In [ ]:
configs_dropout_fine = [
    {'name': 'dropout_0.15', 'hidden': 64, 'n_layers': 1, 'dropout': 0.15, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.25', 'hidden': 64, 'n_layers': 1, 'dropout': 0.25, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'dropout_0.4',  'hidden': 64, 'n_layers': 1, 'dropout': 0.4,  'lr': 0.001, 'batch_size': 32, 'epochs': 40},
]

In [ ]:
best_val_acc_fine = 0
best_run_name_fine = None
dropout_fine_results = []

for cfg in configs_dropout_fine:
    run_name = f"lstm_{cfg['name']}"

    print(f"\n{'-'*60}\nიწყება Fine Dropout ექსპერიმენტი: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'lstm',
            'dropout_included': True,
            'class_weights_included': False,
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_lstm_dropout(
        dropout_rate=cfg['dropout'],
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix (Fine Dropout) — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    model_path = f'{BASE}/models/{run_name}.keras'
    model.save(model_path)

    dropout_fine_results.append({
        'run_name': run_name, 'dropout': cfg['dropout'],
        'train_acc': train_acc_at_best, 'val_acc': val_acc, 'gap': gap,
        'val_loss': val_loss, 'val_f1_macro': report_val['macro avg']['f1-score']
    })

    if val_acc > best_val_acc_fine:
        best_val_acc_fine = val_acc
        best_run_name_fine = run_name

    print(f"✨ Run {run_name} დასრულდა. Val Accuracy: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

dropout_fine_df = pd.DataFrame(dropout_fine_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nFine Dropout sweep დასრულდა!\n{'='*60}")
print(dropout_fine_df[['run_name', 'dropout', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])
print(f"\nსაუკეთესო: {best_run_name_fine} (Val Acc: {best_val_acc_fine:.4f})")



------------------------------------------------------------
იწყება Fine Dropout ექსპერიმენტი: lstm_dropout_0.15
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 61ms/step - accuracy: 0.3983 - loss: 2.1256 - val_accuracy: 0.4328 - val_loss: 1.7942 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 46ms/step - accuracy: 0.6019 - loss: 1.2781 - val_accuracy: 0.5399 - val_loss: 1.5227 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 60ms/step - accuracy: 0.6917 - loss: 0.9882 - val_accuracy: 0.5563 - val_loss: 1.4670 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.7388 - loss: 0.8271 - val_accuracy: 0.5953 - val_loss: 1.4114 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.7774 - loss: 0.7199 - val_accuracy: 0.6107 - val_loss: 1.3680 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.8037 - loss: 0.6356 - val_accuracy: 0.6080 - val_loss: 1.3880 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.8224 - loss

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_dropout_0.15 დასრულდა. Val Accuracy: 0.6733, Gap: 0.2141


epoch/accuracy,▁▄▅▆▆▆▇▇▇▇▇▇█████████
epoch/epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
epoch/learning_rate,████████████▄▄▄▄▂▂▂▂▁
epoch/loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▆▆▆▇█▇██▇█████████
epoch/val_loss,█▄▃▂▁▂▁▁▂▂▃▃▂▃▃▃▃▃▄▃▄
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 64ms/step - accuracy: 0.2861 - loss: 2.4736 - val_accuracy: 0.3766 - val_loss: 2.0485 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.4460 - loss: 1.8019 - val_accuracy: 0.4492 - val_loss: 1.7774 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.5139 - loss: 1.5182 - val_accuracy: 0.4701 - val_loss: 1.6767 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - accuracy: 0.5718 - loss: 1.3239 - val_accuracy: 0.4819 - val_loss: 1.6095 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 54ms/step - accuracy: 0.6116 - loss: 1.2057 - val_accuracy: 0.4955 - val_loss: 1.5887 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - accuracy: 0.6291 - loss: 1.1322 - val_accuracy: 0.5181 - val_loss: 1.5575 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - accuracy: 0.6573 - loss

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

✨ Run lstm_dropout_0.4 დასრულდა. Val Accuracy: 0.6797, Gap: 0.1425


epoch/accuracy,▁▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████████████
epoch/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,████████████▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
epoch/loss,█▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▃▃▄▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇██▇▇████████████████
epoch/val_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Fine Dropout sweep დასრულდა!
            run_name  dropout  train_acc   val_acc       gap  val_loss  \
2   lstm_dropout_0.4     0.40   0.822190  0.679673  0.142517  1.424770   
0  lstm_dropout_0.15     0.15   0.887435  0.673321  0.214114  1.429619   
1  lstm_dropout_0.25     0.25   0.855171  0.653358  0.201814  1.658650   

   val_f1_macro  
2      0.562086  
0      0.578961  
1      0.541777  

საუკეთესო: lstm_dropout_0.4 (Val Acc: 0.6797)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from wandb.integration.keras import WandbMetricsLogger

config_target = {
    'name': 'dropout_0.45',
    'hidden': 64,
    'n_layers': 1,
    'dropout': 0.45,
    'lr': 0.001,
    'batch_size': 32,
    'epochs': 40
}

run_name = f"lstm_{config_target['name']}"

print(f"\n{'-'*60}\nStarting targeted Dropout experiment: {run_name}\n{'-'*60}")

wandb.init(
    project='ildolcefarniente',
    group='p2_lstm',
    name=run_name,
    config={
        **config_target,
        'model': 'lstm',
        'dropout_included': True,
        'class_weights_included': False,
        'seq_len': seq_len,
        'n_features': n_features,
        'n_classes': n_classes,
        'pipeline': 'pipeline2A_33kp',
        'normalization': 'hip_centered_shoulder_hip_scaled'
    },
    reinit=True
)

model = build_lstm_dropout(
    dropout_rate=config_target['dropout'],
    hidden=config_target['hidden'],
    n_layers=config_target['n_layers'],
    lr=config_target['lr']
)

callbacks = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config_target['epochs'],
    batch_size=config_target['batch_size'],
    callbacks=callbacks,
    verbose=1
)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

best_epoch_idx = np.argmax(history.history['val_accuracy'])
train_acc_at_best = history.history['accuracy'][best_epoch_idx]
gap = train_acc_at_best - val_acc

y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

report_val = classification_report(
    y_val_idx, y_pred_val,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True
)

cm_val = confusion_matrix(y_val_idx, y_pred_val)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
ax.set_ylabel('True Val Label')
ax.set_xlabel('Predicted Val Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_path = f'/content/{run_name}_val_cm.png'
plt.savefig(cm_path, dpi=150)
plt.close()

wandb.log({
    'final_val_accuracy': val_acc,
    'final_val_loss': val_loss,
    'train_acc_at_best': train_acc_at_best,
    'overfitting_gap': gap,
    'val_f1_macro': report_val['macro avg']['f1-score'],
    'val_f1_weighted': report_val['weighted avg']['f1-score'],
    'val_confusion_matrix_db': wandb.plot.confusion_matrix(
        probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
    ),
    'val_confusion_matrix_img': wandb.Image(cm_path)
})

print(f"\nRun {run_name} completed.")
print(f"Train Accuracy: {train_acc_at_best:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")
print(f"Gap:            {gap:.4f}")
print(f"Val F1 Macro:   {report_val['macro avg']['f1-score']:.4f}")

wandb.finish()


------------------------------------------------------------
Starting targeted Dropout experiment: lstm_dropout_0.45
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.2556 - loss: 2.5937 - val_accuracy: 0.3648 - val_loss: 2.1231 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 60ms/step - accuracy: 0.4132 - loss: 1.9360 - val_accuracy: 0.4446 - val_loss: 1.8358 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 46ms/step - accuracy: 0.4779 - loss: 1.6432 - val_accuracy: 0.4791 - val_loss: 1.6704 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 60ms/step - accuracy: 0.5318 - loss: 1.4715 - val_accuracy: 0.4646 - val_loss: 1.6599 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.5643 - loss: 1.3350 - val_accuracy: 0.5073 - val_loss: 1.6211 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.5935 - loss: 1.2581 - val_accuracy: 0.5118 - val_loss: 1.5877 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.6214 - los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me


Run lstm_dropout_0.45 completed.
Train Accuracy: 0.7517
Val Accuracy:   0.6216
Gap:            0.1302
Val F1 Macro:   0.4959


epoch/accuracy,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇█████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epoch/learning_rate,█████████████████▃▃▃▃▃▁▁▁▁
epoch/loss,█▆▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▄▅▅▅▆▆▆▆▇█▇█▇▇█████████
epoch/val_loss,█▅▃▃▃▃▂▂▂▂▁▂▁▁▁▂▁▁▁▁▂▂▁▁▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from wandb.integration.keras import WandbMetricsLogger

# 2-Layer (Stacked) LSTM Builder
def build_lstm_stacked(config):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        # First layer passes sequences to the second layer
        LSTM(config['hidden_1'], return_sequences=True, dropout=config['dropout']),
        # Second layer processes the sequences into a single vector
        LSTM(config['hidden_2'], dropout=config['dropout']),

        Dense(64, activation='relu'),
        Dropout(config['dropout']),
        Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=config['lr']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

config_stacked = {
    'name': 'stacked_lstm_opt',
    'hidden_1': 64,    # First LSTM layer units
    'hidden_2': 32,    # Second LSTM layer units
    'dropout': 0.40,   # Using the best dropout from previous phase
    'lr': 0.001,
    'batch_size': 32,
    'epochs': 50       # Giving it slightly more epochs to converge
}

run_name = f"lstm_{config_stacked['name']}"

print(f"\n{'-'*60}\nStarting Stacked LSTM experiment: {run_name}\n{'-'*60}")

wandb.init(
    project='ildolcefarniente',
    group='p2_lstm',
    name=run_name,
    config={
        **config_stacked,
        'model': 'lstm',
        'architecture': 'stacked-lstm',
        'seq_len': seq_len,
        'n_features': n_features,
        'n_classes': n_classes,
        'pipeline': 'pipeline2A_33kp',
        'normalization': 'hip_centered_shoulder_hip_scaled'
    },
    reinit=True
)

model = build_lstm_stacked(config_stacked)

callbacks = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config_stacked['epochs'],
    batch_size=config_stacked['batch_size'],
    callbacks=callbacks,
    verbose=1
)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

best_epoch_idx = np.argmax(history.history['val_accuracy'])
train_acc_at_best = history.history['accuracy'][best_epoch_idx]
gap = train_acc_at_best - val_acc

y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

report_val = classification_report(
    y_val_idx, y_pred_val,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True
)

cm_val = confusion_matrix(y_val_idx, y_pred_val)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
ax.set_ylabel('True Val Label')
ax.set_xlabel('Predicted Val Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_path = f'/content/{run_name}_val_cm.png'
plt.savefig(cm_path, dpi=150)
plt.close()

wandb.log({
    'final_val_accuracy': val_acc,
    'final_val_loss': val_loss,
    'train_acc_at_best': train_acc_at_best,
    'overfitting_gap': gap,
    'val_f1_macro': report_val['macro avg']['f1-score'],
    'val_f1_weighted': report_val['weighted avg']['f1-score'],
    'val_confusion_matrix_db': wandb.plot.confusion_matrix(
        probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
    ),
    'val_confusion_matrix_img': wandb.Image(cm_path)
})

print(f"\nRun {run_name} completed.")
print(f"Train Accuracy: {train_acc_at_best:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")
print(f"Gap:            {gap:.4f}")
print(f"Val F1 Macro:   {report_val['macro avg']['f1-score']:.4f}")

wandb.finish()


------------------------------------------------------------
Starting Stacked LSTM experiment: lstm_stacked_lstm_opt
------------------------------------------------------------


Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.2497 - loss: 2.6337 - val_accuracy: 0.3593 - val_loss: 2.0997 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4062 - loss: 1.9430 - val_accuracy: 0.4465 - val_loss: 1.8480 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4736 - loss: 1.6847 - val_accuracy: 0.4701 - val_loss: 1.7513 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5150 - loss: 1.5193 - val_accuracy: 0.4855 - val_loss: 1.6615 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5530 - loss: 1.4133 - val_accuracy: 0.5009 - val_loss: 1.6068 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5747 - loss: 1.3294 - val_accuracy: 0.5145 - val_loss: 1.6193 - learning_rate: 0.0010
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.5994 - loss: 1.267

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me


Run lstm_stacked_lstm_opt completed.
Train Accuracy: 0.7311
Val Accuracy:   0.6016
Gap:            0.1295
Val F1 Macro:   0.4518


epoch/accuracy,▁▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████▃▃▃▃▃▁▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▄▅▅▅▆▇▆▇▇▇▇▇█▇▇█████▇████████
epoch/val_loss,█▅▄▃▃▃▃▂▂▂▂▂▁▁▂▁▂▂▂▁▁▂▂▁▁▁▁▁▂▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from wandb.integration.keras import WandbMetricsLogger

# 1. Calculate class weights automatically
# Converting one-hot encoded y_train to 1D array of class indices
y_train_indices = np.argmax(y_train, axis=1)
unique_classes = np.unique(y_train_indices)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train_indices
)

# Transforming into a dictionary format that Keras .fit() expects
class_weight_dict = dict(zip(unique_classes, class_weights_array))

print("Calculated Class Weights:")
for cls_idx, weight in class_weight_dict.items():
    print(f"Class {cls_idx} ({classes[cls_idx]}): {weight:.4f}")

# 2. Setup the best 1-layer architecture config
config_weights = {
    'name': 'lstm_with_class_weights',
    'hidden': 64,
    'dropout': 0.40,
    'lr': 0.001,
    'batch_size': 32,
    'epochs': 40
}

run_name = config_weights['name']

print(f"\n{'-'*60}\nStarting Experiment with Class Weights: {run_name}\n{'-'*60}")

wandb.init(
    project='ildolcefarniente',
    group='p2_lstm',
    name=run_name,
    config={
        **config_weights,
        'model': 'lstm',
        'dropout_included': True,
        'class_weights_included': True,
        'seq_len': seq_len,
        'n_features': n_features,
        'n_classes': n_classes,
        'pipeline': 'pipeline2A_33kp',
        'normalization': 'hip_centered_shoulder_hip_scaled'
    },
    reinit=True
)

# Rebuilding the champion 1-layer model
model = Sequential([
    Input(shape=(seq_len, n_features)),
    LSTM(config_weights['hidden']),
    Dropout(config_weights['dropout']),
    Dense(n_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=config_weights['lr']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

# Passing the class_weight dictionary to .fit()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config_weights['epochs'],
    batch_size=config_weights['batch_size'],
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

best_epoch_idx = np.argmax(history.history['val_accuracy'])
train_acc_at_best = history.history['accuracy'][best_epoch_idx]
gap = train_acc_at_best - val_acc

y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

report_val = classification_report(
    y_val_idx, y_pred_val,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True
)

cm_val = confusion_matrix(y_val_idx, y_pred_val)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
ax.set_ylabel('True Val Label')
ax.set_xlabel('Predicted Val Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_path = f'/content/{run_name}_val_cm.png'
plt.savefig(cm_path, dpi=150)
plt.close()

wandb.log({
    'final_val_accuracy': val_acc,
    'final_val_loss': val_loss,
    'train_acc_at_best': train_acc_at_best,
    'overfitting_gap': gap,
    'val_f1_macro': report_val['macro avg']['f1-score'],
    'val_f1_weighted': report_val['weighted avg']['f1-score'],
    'val_confusion_matrix_db': wandb.plot.confusion_matrix(
        probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
    ),
    'val_confusion_matrix_img': wandb.Image(cm_path)
})

print(f"\nRun {run_name} completed.")
print(f"Train Accuracy: {train_acc_at_best:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")
print(f"Gap:            {gap:.4f}")
print(f"Val F1 Macro:   {report_val['macro avg']['f1-score']:.4f}")

wandb.finish()

Calculated Class Weights:
Class 0 (bench_press): 1.1623
Class 1 (bicep_curl): 1.3525
Class 2 (chest_fly): 1.9074
Class 3 (clean_and_jerk): 0.3407
Class 4 (deadlift): 1.8912
Class 5 (decline_bench_press): 2.8610
Class 6 (hammer_curl): 1.3363
Class 7 (hip_thrust): 1.7997
Class 8 (incline_bench_press): 1.3443
Class 9 (jump_rope): 9.7026
Class 11 (lat_pulldown): 1.4779
Class 12 (lateral_raise): 0.9298
Class 13 (leg_extension): 1.3525
Class 14 (leg_raises): 1.2608
Class 15 (plank): 0.5510
Class 16 (pullup): 0.5951
Class 17 (pushup): 0.6340
Class 18 (romanian_deadlift): 1.4779
Class 19 (russian_twist): 1.4586
Class 20 (shoulder_press): 1.4305
Class 21 (situp): 1.3127
Class 22 (squat): 0.3287
Class 23 (t_bar_row): 0.9537
Class 24 (tricep_dips): 1.0098
Class 25 (tricep_pushdown): 1.5390

------------------------------------------------------------
Starting Experiment with Class Weights: lstm_with_class_weights
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.3895 - loss: 2.2921 - val_accuracy: 0.3966 - val_loss: 1.9533 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6207 - loss: 1.3924 - val_accuracy: 0.5064 - val_loss: 1.5911 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7039 - loss: 1.0515 - val_accuracy: 0.5526 - val_loss: 1.5144 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7458 - loss: 0.8686 - val_accuracy: 0.5417 - val_loss: 1.4874 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7785 - loss: 0.7395 - val_accuracy: 0.5880 - val_loss: 1.4195 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8032 - loss: 0.6468 - val_accuracy: 0.6053 - val_loss: 1.3986 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8129 - loss: 0.602

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me


Run lstm_with_class_weights completed.
Train Accuracy: 0.8883
Val Accuracy:   0.6407
Gap:            0.2477
Val F1 Macro:   0.5380


epoch/accuracy,▁▄▅▆▆▆▆▇▇▇▇████████
epoch/epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
epoch/learning_rate,██████████▃▃▃▃▃▁▁▁▁
epoch/loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▅▆▇▇▆█▇███▇█████
epoch/val_loss,█▄▃▂▁▁▁▂▂▂▁▂▂▃▂▂▂▂▂
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


In [ ]:
import numpy as np

# Convert to indices
y_train_idx = np.argmax(y_train, axis=1)
y_val_idx = np.argmax(y_val, axis=1)

print(f"Total samples in Train: {len(y_train_idx)}")
print(f"Total samples in Val:   {len(y_val_idx)}\n")

print("Class counts in Train:")
for i in range(n_classes):
    count = np.sum(y_train_idx == i)
    if count == 0:
        print(f"❌ Class {i} ({classes[i] if i < len(classes) else 'Unknown'}) has ZERO samples in TRAIN!")
    else:
        print(f"Class {i} ({classes[i] if i < len(classes) else 'Unknown'}): {count} samples")

print("\n" + "="*40 + "\n")

print("Class counts in Val:")
for i in range(n_classes):
    count = np.sum(y_val_idx == i)
    if count == 0:
        print(f"⚠️ Class {i} ({classes[i] if i < len(classes) else 'Unknown'}) has ZERO samples in VAL.")
    else:
        print(f"Class {i} ({classes[i] if i < len(classes) else 'Unknown'}): {count} samples")

Total samples in Train: 5579
Total samples in Val:   1102

Class counts in Train:
Class 0 (bench_press): 192 samples
Class 1 (bicep_curl): 165 samples
Class 2 (chest_fly): 117 samples
Class 3 (clean_and_jerk): 655 samples
Class 4 (deadlift): 118 samples
Class 5 (decline_bench_press): 78 samples
Class 6 (hammer_curl): 167 samples
Class 7 (hip_thrust): 124 samples
Class 8 (incline_bench_press): 166 samples
Class 9 (jump_rope): 23 samples
❌ Class 10 (jumping_jacks) has ZERO samples in TRAIN!
Class 11 (lat_pulldown): 151 samples
Class 12 (lateral_raise): 240 samples
Class 13 (leg_extension): 165 samples
Class 14 (leg_raises): 177 samples
Class 15 (plank): 405 samples
Class 16 (pullup): 375 samples
Class 17 (pushup): 352 samples
Class 18 (romanian_deadlift): 151 samples
Class 19 (russian_twist): 153 samples
Class 20 (shoulder_press): 156 samples
Class 21 (situp): 170 samples
Class 22 (squat): 679 samples
Class 23 (t_bar_row): 234 samples
Class 24 (tricep_dips): 221 samples
Class 25 (tricep_

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from wandb.integration.keras import WandbMetricsLogger

# 1. THE DEFINITION (Fixes the NameError)
def build_lstm_with_dropout(hidden=64, n_layers=1, dropout_rate=0.4, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.LSTM(
            hidden,
            return_sequences=return_seq,
            dropout=dropout_rate,
            recurrent_dropout=dropout_rate
        ))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 2. THE EXPERIMENT GRID (Testing 64 and 128 units with 0.4 dropout)
configs_rec_grid = [
    {'name': 'rec_drop_0.4_h64',  'hidden': 64,  'dropout_rate': 0.40, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'rec_drop_0.4_h128', 'hidden': 128, 'dropout_rate': 0.40, 'lr': 0.001, 'batch_size': 32, 'epochs': 40}
]

grid_results = []

for cfg in configs_rec_grid:
    run_name = f"lstm_{cfg['name']}"

    print(f"\n{'-'*60}\nStarting Experiment: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'lstm',
            'recurrent_dropout_included': True,
            'n_layers': 1,
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    # Calling the function defined right above
    model = build_lstm_with_dropout(
        hidden=cfg['hidden'],
        n_layers=1,
        dropout_rate=cfg['dropout_rate'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    grid_results.append({
        'run_name': run_name,
        'hidden': cfg['hidden'],
        'dropout': cfg['dropout_rate'],
        'train_acc': train_acc_at_best,
        'val_acc': val_acc,
        'gap': gap,
        'val_loss': val_loss,
        'val_f1_macro': report_val['macro avg']['f1-score']
    })

    print(f"Run {run_name} finished. Val Acc: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

# 3. DISPLAY THE SUMMARY
rec_dropout_df = pd.DataFrame(grid_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nRecurrent Dropout Sweep Completed!\n{'='*60}")
print(rec_dropout_df[['run_name', 'hidden', 'dropout', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])


------------------------------------------------------------
Starting Experiment: lstm_rec_drop_0.4_h64
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 64ms/step - accuracy: 0.2909 - loss: 2.4468 - val_accuracy: 0.3875 - val_loss: 2.0567 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - accuracy: 0.4361 - loss: 1.8008 - val_accuracy: 0.4474 - val_loss: 1.7307 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.5151 - loss: 1.5096 - val_accuracy: 0.4982 - val_loss: 1.6530 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.5712 - loss: 1.3336 - val_accuracy: 0.5290 - val_loss: 1.5576 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 52ms/step - accuracy: 0.6110 - loss: 1.2203 - val_accuracy: 0.5454 - val_loss: 1.5168 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 10s 58ms/step - accuracy: 0.6369 - loss: 1.1179 - val_accuracy: 0.5762 - val_loss: 1.5374 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.6591 - los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run lstm_rec_drop_0.4_h64 finished. Val Acc: 0.6307, Gap: 0.1372


epoch/accuracy,▁▃▄▅▆▆▆▇▇▇▇▇▇▇███████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,█████████▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▅▆▆▆▇▇▇▇▇▇████████████████████
epoch/val_loss,█▄▃▂▁▁▁▁▁▁▁▁▁▂▁▂▂▂▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - accuracy: 0.3420 - loss: 2.2928 - val_accuracy: 0.4410 - val_loss: 1.8048 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.5060 - loss: 1.5719 - val_accuracy: 0.5154 - val_loss: 1.6014 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.5804 - loss: 1.3152 - val_accuracy: 0.5581 - val_loss: 1.5015 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.6422 - loss: 1.1354 - val_accuracy: 0.5917 - val_loss: 1.4671 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.6713 - loss: 1.0302 - val_accuracy: 0.5771 - val_loss: 1.4656 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - accuracy: 0.6983 - loss: 0.9437 - val_accuracy: 0.6044 - val_loss: 1.4070 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.7249 - los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run lstm_rec_drop_0.4_h128 finished. Val Acc: 0.6615, Gap: 0.1727


epoch/accuracy,▁▃▄▅▆▆▆▇▇▇▇▇▇▇█████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,██████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▅▆▅▆▆▇▇▇▇▇▇▇▇████████████████
epoch/val_loss,█▄▃▂▂▁▂▁▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Recurrent Dropout Sweep Completed!
                 run_name  hidden  dropout  train_acc   val_acc       gap  \
1  lstm_rec_drop_0.4_h128     128      0.4    0.83420  0.661524  0.172675   
0   lstm_rec_drop_0.4_h64      64      0.4    0.76788  0.630672  0.137208   

   val_loss  val_f1_macro  
1  1.461676      0.528821  
0  1.549284      0.497041  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from wandb.integration.keras import WandbMetricsLogger

# 1. Bidirectional LSTM Builder
def build_bidirectional_lstm(hidden=64, dropout_rate=0.4, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    # Wrapping the LSTM layer in a Bidirectional wrapper
    model.add(layers.Bidirectional(layers.LSTM(hidden, dropout=dropout_rate)))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 2. Smart Bidirectional Grid Configs (No Class Weights to prevent overfitting)
configs_bi_grid = [
    {'name': 'bi_lstm_h64_b32',  'hidden': 64,  'dropout': 0.40, 'lr': 0.001,  'batch_size': 32, 'epochs': 40},
    {'name': 'bi_lstm_h64_b64',  'hidden': 64,  'dropout': 0.40, 'lr': 0.001,  'batch_size': 64, 'epochs': 40},
    {'name': 'bi_lstm_h128_b32', 'hidden': 128, 'dropout': 0.40, 'lr': 0.001,  'batch_size': 32, 'epochs': 40},
    {'name': 'bi_lstm_h128_b64', 'hidden': 128, 'dropout': 0.40, 'lr': 0.0005, 'batch_size': 64, 'epochs': 40} # smaller LR for bigger batch/units
]

bi_results = []

for cfg in configs_bi_grid:
    run_name = cfg['name']

    print(f"\n{'-'*60}\nStarting Bidirectional Experiment: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'bidirectional_lstm',
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_bidirectional_lstm(
        hidden=cfg['hidden'],
        dropout_rate=cfg['dropout'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    bi_results.append({
        'run_name': run_name,
        'hidden': cfg['hidden'],
        'batch_size': cfg['batch_size'],
        'train_acc': train_acc_at_best,
        'val_acc': val_acc,
        'gap': gap,
        'val_loss': val_loss,
        'val_f1_macro': report_val['macro avg']['f1-score']
    })

    print(f"Run {run_name} finished. Val Acc: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

# Summary Dataframe
bi_df = pd.DataFrame(bi_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nBidirectional LSTM Sweep Completed!\n{'='*60}")
print(bi_df[['run_name', 'hidden', 'batch_size', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])


------------------------------------------------------------
Starting Bidirectional Experiment: bi_lstm_h64_b32
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.3402 - loss: 2.2626 - val_accuracy: 0.4583 - val_loss: 1.7731 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5255 - loss: 1.5264 - val_accuracy: 0.4927 - val_loss: 1.5506 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5989 - loss: 1.2516 - val_accuracy: 0.5644 - val_loss: 1.4490 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.6394 - loss: 1.1062 - val_accuracy: 0.5817 - val_loss: 1.3943 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6930 - loss: 0.9622 - val_accuracy: 0.5780 - val_loss: 1.4798 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7080 - loss: 0.8914 - val_accuracy: 0.6252 - val_loss: 1.4275 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7302 - loss: 0.834

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run bi_lstm_h64_b32 finished. Val Acc: 0.6688, Gap: 0.1597


epoch/accuracy,▁▄▅▅▆▆▆▇▇▇▇▇███████████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▅▅▅▇▇▆▇▇▇▇███████████████
epoch/val_loss,█▄▂▁▃▂▁▂▂▂▂▂▂▂▂▂▂▂▃▃▂▂▃▃▃▂▃
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.2952 - loss: 2.5206 - val_accuracy: 0.3956 - val_loss: 2.0779 - learning_rate: 5.0000e-04
Epoch 2/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4601 - loss: 1.8253 - val_accuracy: 0.4673 - val_loss: 1.7721 - learning_rate: 5.0000e-04
Epoch 3/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5393 - loss: 1.5120 - val_accuracy: 0.5163 - val_loss: 1.6001 - learning_rate: 5.0000e-04
Epoch 4/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5981 - loss: 1.3074 - val_accuracy: 0.5517 - val_loss: 1.4681 - learning_rate: 5.0000e-04
Epoch 5/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6474 - loss: 1.1422 - val_accuracy: 0.5744 - val_loss: 1.4321 - learning_rate: 5.0000e-04
Epoch 6/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6673 - loss: 1.0516 - val_accuracy: 0.6007 - val_loss: 1.3799 - learning_rate: 5.0000e-04
Epoch 7/40
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6969 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run bi_lstm_h128_b64 finished. Val Acc: 0.6652, Gap: 0.1597


epoch/accuracy,▁▃▄▅▆▆▆▇▇▇▇▇▇▇███████████████████████
epoch/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
epoch/learning_rate,██████████▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▅▆▆▇▇▇▇▇▇▇▇▇██████████████████████
epoch/val_loss,█▅▃▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Bidirectional LSTM Sweep Completed!
           run_name  hidden  batch_size  train_acc   val_acc       gap  \
2  bi_lstm_h128_b32     128          32   0.822370  0.679673  0.142696   
0   bi_lstm_h64_b32      64          32   0.828464  0.668784  0.159680   
3  bi_lstm_h128_b64     128          64   0.824879  0.665154  0.159725   
1   bi_lstm_h64_b64      64          64   0.737587  0.646098  0.091489   

   val_loss  val_f1_macro  
2  1.431040      0.595749  
0  1.477725      0.552926  
3  1.411312      0.549199  
1  1.326834      0.535779  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from wandb.integration.keras import WandbMetricsLogger

# 1. Stacked Bidirectional LSTM Builder (No CNN)
def build_stacked_bilstm(hidden=64, n_layers=2, dropout_rate=0.4, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    # Building stacked BiLSTM layers
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)
        model.add(layers.Bidirectional(
            layers.LSTM(hidden, dropout=dropout_rate, return_sequences=return_seq)
        ))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 2. Configs for Stacked BiLSTM Sweep
configs_stacked_bi = [
    {'name': 'stacked_bi_h64_l2',  'hidden': 64,  'n_layers': 2, 'dropout': 0.40, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'stacked_bi_h128_l2', 'hidden': 128, 'n_layers': 2, 'dropout': 0.40, 'lr': 0.0005, 'batch_size': 32, 'epochs': 40} # slightly lower LR for big stacked model
]

stacked_bi_results = []

for cfg in configs_stacked_bi:
    run_name = cfg['name']

    print(f"\n{'-'*60}\nStarting Stacked BiLSTM Experiment: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': 'stacked_bidirectional_lstm',
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_stacked_bilstm(
        hidden=cfg['hidden'],
        n_layers=cfg['n_layers'],
        dropout_rate=cfg['dropout'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    stacked_bi_results.append({
        'run_name': run_name,
        'hidden': cfg['hidden'],
        'layers': cfg['n_layers'],
        'train_acc': train_acc_at_best,
        'val_acc': val_acc,
        'gap': gap,
        'val_loss': val_loss,
        'val_f1_macro': report_val['macro avg']['f1-score']
    })

    print(f"Run {run_name} finished. Val Acc: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

# Summary Table
stacked_bi_df = pd.DataFrame(stacked_bi_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nStacked Bidirectional Sweep Completed!\n{'='*60}")
print(stacked_bi_df[['run_name', 'hidden', 'layers', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])


------------------------------------------------------------
Starting Stacked BiLSTM Experiment: stacked_bi_h64_l2
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.3237 - loss: 2.3025 - val_accuracy: 0.4519 - val_loss: 1.8386 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4958 - loss: 1.5845 - val_accuracy: 0.4809 - val_loss: 1.6426 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5662 - loss: 1.3424 - val_accuracy: 0.5481 - val_loss: 1.4806 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6046 - loss: 1.2137 - val_accuracy: 0.5499 - val_loss: 1.4745 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6428 - loss: 1.0888 - val_accuracy: 0.5726 - val_loss: 1.4312 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6854 - loss: 0.9794 - val_accuracy: 0.5735 - val_loss: 1.4157 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6903 - loss: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run stacked_bi_h64_l2 finished. Val Acc: 0.6497, Gap: 0.1698


epoch/accuracy,▁▃▄▅▅▆▆▆▇▇▇▇▇▇▇████████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,█████████████▄▄▄▄▂▂▂▂▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▄▄▅▅▆▇▇▇▇▇▇██▇▇██████████
epoch/val_loss,█▅▃▃▃▂▃▂▁▂▄▄▃▃▃▄▄▄▄▄▄▄▄▄▃▃▃
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.3416 - loss: 2.3125 - val_accuracy: 0.4483 - val_loss: 1.8917 - learning_rate: 5.0000e-04
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5137 - loss: 1.5856 - val_accuracy: 0.4955 - val_loss: 1.6651 - learning_rate: 5.0000e-04
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5917 - loss: 1.3022 - val_accuracy: 0.5644 - val_loss: 1.5422 - learning_rate: 5.0000e-04
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6333 - loss: 1.1380 - val_accuracy: 0.5880 - val_loss: 1.5029 - learning_rate: 5.0000e-04
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.6802 - loss: 1.0252 - val_accuracy: 0.6053 - val_loss: 1.5005 - learning_rate: 5.0000e-04
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7051 - loss: 0.9448 - val_accuracy: 0.6180 - val_loss: 1.4484 - learning_rate: 5.0000e-04
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - acc

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run stacked_bi_h128_l2 finished. Val Acc: 0.6679, Gap: 0.1823


epoch/accuracy,▁▃▄▅▆▆▆▇▇▇▇▇████████████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,██████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▅▅▆▆▇▆▆▇▇▇▇▇███▇██████████
epoch/val_loss,█▄▂▂▂▁▂▃▃▂▃▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Stacked Bidirectional Sweep Completed!
             run_name  hidden  layers  train_acc   val_acc       gap  \
1  stacked_bi_h128_l2     128       2   0.850152  0.667877  0.182276   
0   stacked_bi_h64_l2      64       2   0.819502  0.649728  0.169774   

   val_loss  val_f1_macro  
1  1.556310      0.553675  
0  1.516176      0.517232  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from wandb.integration.keras import WandbMetricsLogger

# Architecture with SpatialDropout1D
def build_spatial_bilstm(hidden=128, dropout_rate=0.4, spatial_dropout_rate=0.3, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    # Drops entire 1D feature channels to prevent overfitting on specific joints
    model.add(layers.SpatialDropout1D(spatial_dropout_rate))

    # Our champion 1-layer BiLSTM
    model.add(layers.Bidirectional(layers.LSTM(hidden, dropout=dropout_rate)))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

run_name = "bilstm_spatial_dropout_0.3"

print(f"\n{'-'*60}\nStarting Spatial Dropout Experiment: {run_name}\n{'-'*60}")

wandb.init(
    project='ildolcefarniente',
    group='p2_lstm',
    name=run_name,
    config={
        'hidden': 128,
        'dropout': 0.40,
        'spatial_dropout': 0.30,
        'lr': 0.001,
        'batch_size': 32,
        'epochs': 40,
        'model': 'spatial_bilstm',
        'seq_len': seq_len,
        'n_features': n_features,
        'n_classes': n_classes,
        'pipeline': 'pipeline2A_33kp',
        'normalization': 'hip_centered_shoulder_hip_scaled'
    },
    reinit=True
)

model = build_spatial_bilstm(hidden=128, dropout_rate=0.40, spatial_dropout_rate=0.30, lr=0.001)

callbacks = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

best_epoch_idx = np.argmax(history.history['val_accuracy'])
train_acc_at_best = history.history['accuracy'][best_epoch_idx]
gap = train_acc_at_best - val_acc

y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

report_val = classification_report(
    y_val_idx, y_pred_val,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True
)

cm_val = confusion_matrix(y_val_idx, y_pred_val)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
ax.set_ylabel('True Val Label')
ax.set_xlabel('Predicted Val Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_path = f'/content/{run_name}_val_cm.png'
plt.savefig(cm_path, dpi=150)
plt.close()

wandb.log({
    'final_val_accuracy': val_acc,
    'final_val_loss': val_loss,
    'train_acc_at_best': train_acc_at_best,
    'overfitting_gap': gap,
    'val_f1_macro': report_val['macro avg']['f1-score'],
    'val_f1_weighted': report_val['weighted avg']['f1-score'],
    'val_confusion_matrix_db': wandb.plot.confusion_matrix(
        probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
    ),
    'val_confusion_matrix_img': wandb.Image(cm_path)
})

print(f"\nRun {run_name} completed.")
print(f"Train Accuracy: {train_acc_at_best:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")
print(f"Gap:            {gap:.4f}")
print(f"Val F1 Macro:   {report_val['macro avg']['f1-score']:.4f}")

wandb.finish()


------------------------------------------------------------
Starting Spatial Dropout Experiment: bilstm_spatial_dropout_0.3
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.3461 - loss: 2.2214 - val_accuracy: 0.4356 - val_loss: 1.8371 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.5013 - loss: 1.5767 - val_accuracy: 0.5054 - val_loss: 1.6377 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5678 - loss: 1.3307 - val_accuracy: 0.5572 - val_loss: 1.5331 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6170 - loss: 1.1880 - val_accuracy: 0.5163 - val_loss: 1.6576 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6494 - loss: 1.0808 - val_accuracy: 0.5644 - val_loss: 1.4884 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6752 - loss: 0.9967 - val_accuracy: 0.5717 - val_loss: 1.5255 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.6987 - loss: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me


Run bilstm_spatial_dropout_0.3 completed.
Train Accuracy: 0.8129
Val Accuracy:   0.6361
Gap:            0.1768
Val F1 Macro:   0.5175


epoch/accuracy,▁▃▄▅▅▆▆▆▇▇▇▇▇▇█████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▅▄▅▆▇▆▇▇▆▇██▇████████████████
epoch/val_loss,█▄▂▅▁▂▁▃▂▃▃▂▃▁▂▂▃▃▃▃▃▄▃▃▄▄▄▄▄▄▄
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from wandb.integration.keras import WandbMetricsLogger

# Builder function for both hybrid options
def build_conv_rnn_hybrid(rnn_type='lstm', hidden=64, dropout_rate=0.4, lr=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))

    # 1D Convolution for local temporal feature extraction
    model.add(layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'))
    model.add(layers.MaxPooling1D(pool_size=1))
    model.add(layers.Dropout(dropout_rate))

    # Choosing the recurrent layer
    if rnn_type == 'bilstm':
        model.add(layers.Bidirectional(layers.LSTM(hidden, dropout=dropout_rate)))
    else:
        model.add(layers.LSTM(hidden, dropout=dropout_rate))

    # Dense Classifier Head
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Hybrids to test
configs_hybrid = [
    {'name': 'conv_lstm_h64',   'rnn_type': 'lstm',   'hidden': 64, 'dropout': 0.40, 'lr': 0.001, 'batch_size': 32, 'epochs': 40},
    {'name': 'conv_bilstm_h64', 'rnn_type': 'bilstm', 'hidden': 64, 'dropout': 0.40, 'lr': 0.001, 'batch_size': 32, 'epochs': 40}
]

hybrid_results = []

for cfg in configs_hybrid:
    run_name = cfg['name']

    print(f"\n{'-'*60}\nStarting Hybrid Experiment: {run_name}\n{'-'*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_lstm',
        name=run_name,
        config={
            **cfg,
            'model': f"conv_{cfg['rnn_type']}",
            'seq_len': seq_len,
            'n_features': n_features,
            'n_classes': n_classes,
            'pipeline': 'pipeline2A_33kp',
            'normalization': 'hip_centered_shoulder_hip_scaled'
        },
        reinit=True
    )

    model = build_conv_rnn_hybrid(
        rnn_type=cfg['rnn_type'],
        hidden=cfg['hidden'],
        dropout_rate=cfg['dropout'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    best_epoch_idx = np.argmax(history.history['val_accuracy'])
    train_acc_at_best = history.history['accuracy'][best_epoch_idx]
    gap = train_acc_at_best - val_acc

    y_pred_val = np.argmax(model.predict(X_val, verbose=0), axis=1)

    report_val = classification_report(
        y_val_idx, y_pred_val,
        labels=list(range(len(classes))),
        target_names=classes,
        output_dict=True
    )

    cm_val = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f'Val Confusion Matrix — {run_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Val Label')
    ax.set_xlabel('Predicted Val Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    cm_path = f'/content/{run_name}_val_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_accuracy': val_acc,
        'final_val_loss': val_loss,
        'train_acc_at_best': train_acc_at_best,
        'overfitting_gap': gap,
        'val_f1_macro': report_val['macro avg']['f1-score'],
        'val_f1_weighted': report_val['weighted avg']['f1-score'],
        'val_confusion_matrix_db': wandb.plot.confusion_matrix(
            probs=None, y_true=y_val_idx.tolist(), preds=y_pred_val.tolist(), class_names=classes
        ),
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    hybrid_results.append({
        'run_name': run_name,
        'rnn_type': cfg['rnn_type'],
        'train_acc': train_acc_at_best,
        'val_acc': val_acc,
        'gap': gap,
        'val_loss': val_loss,
        'val_f1_macro': report_val['macro avg']['f1-score']
    })

    print(f"Run {run_name} finished. Val Acc: {val_acc:.4f}, Gap: {gap:.4f}")
    wandb.finish()

# Summary Table
hybrid_df = pd.DataFrame(hybrid_results).sort_values('val_acc', ascending=False)
print(f"\n{'='*60}\nConv-RNN Hybrid Sweep Completed!\n{'='*60}")
print(hybrid_df[['run_name', 'rnn_type', 'train_acc', 'val_acc', 'gap', 'val_loss', 'val_f1_macro']])


------------------------------------------------------------
Starting Hybrid Experiment: conv_lstm_h64
------------------------------------------------------------


Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.2714 - loss: 2.5326 - val_accuracy: 0.3929 - val_loss: 1.9998 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4123 - loss: 1.9121 - val_accuracy: 0.4746 - val_loss: 1.7555 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4752 - loss: 1.6556 - val_accuracy: 0.5191 - val_loss: 1.5979 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5101 - loss: 1.5200 - val_accuracy: 0.5064 - val_loss: 1.5694 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5388 - loss: 1.4523 - val_accuracy: 0.5127 - val_loss: 1.5921 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5582 - loss: 1.3751 - val_accuracy: 0.5181 - val_loss: 1.5446 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5831 - loss: 1.3059

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run conv_lstm_h64 finished. Val Acc: 0.5708, Gap: 0.1414


epoch/accuracy,▁▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▆▅▆▆▇▆▇▆▇▇▇▇█▇▇▇█▇▇████▇█▇▇███
epoch/val_loss,█▅▃▃▃▂▃▃▂▂▁▁▃▃▁▁▂▃▂▃▃▂▃▂▃▃▃▃▃▃▃▃
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...


...

Epoch 1/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.2927 - loss: 2.4156 - val_accuracy: 0.3975 - val_loss: 2.0720 - learning_rate: 0.0010
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4409 - loss: 1.7522 - val_accuracy: 0.4891 - val_loss: 1.8284 - learning_rate: 0.0010
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.5193 - loss: 1.5078 - val_accuracy: 0.5299 - val_loss: 1.6558 - learning_rate: 0.0010
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5716 - loss: 1.3417 - val_accuracy: 0.5327 - val_loss: 1.6349 - learning_rate: 0.0010
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6012 - loss: 1.2507 - val_accuracy: 0.5789 - val_loss: 1.5160 - learning_rate: 0.0010
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.6223 - loss: 1.1750 - val_accuracy: 0.5889 - val_loss: 1.4838 - learning_rate: 0.0010
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.6343 - loss: 1.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Run conv_bilstm_h64 finished. Val Acc: 0.6279, Gap: 0.1437


epoch/accuracy,▁▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇█▇▇█████████
epoch/epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
epoch/learning_rate,██████████████▄▄▄▄▄▄▄▄▂▂▂▂▁▁▁
epoch/loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▅▇▇▇▇▇▆▇▇▇▇▇███████▇██████
epoch/val_loss,█▅▃▃▂▁▂▁▁▁▁▁▂▁▁▁▂▁▂▁▁▂▂▁▂▂▂▂▂
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,▁
train_acc_at_best,▁
+2,...



Conv-RNN Hybrid Sweep Completed!
          run_name rnn_type  train_acc   val_acc       gap  val_loss  \
1  conv_bilstm_h64   bilstm   0.771644  0.627949  0.143694  1.457039   
0    conv_lstm_h64     lstm   0.712135  0.570780  0.141354  1.554115   

   val_f1_macro  
1      0.533045  
0      0.431509  


In [ ]:
# =====================================================================
# Cell: საუკეთესო LSTM მოდელის ავტომატური შენახვა Drive-ზე და WandB-ზე
# =====================================================================
import os
import wandb

# 1. განვსაზღვროთ ჩვენი გამარჯვებული მოდელის მონაცემები
best_run_name = "lstm_dropout_0.4"
best_val_f1_macro = 0.5621  # შენი მოდელის ზუსტი F1 ქულა
best_val_acc = 0.6797       # ვალიდაციის სიზუსტე
overfitting_gap = 0.1425    # train_acc - val_acc

print(f"საუკეთესო შერჩეული მოდელი: {best_run_name}")
print(f"Validation Accuracy: {best_val_acc:.4f}")
print(f"Validation F1-Macro: {best_val_f1_macro:.4f}")
print(f"Overfitting Gap: {overfitting_gap:.4f}")

# 2. ვქმნით საქაღალდეს Google Drive-ზე
models_dir = '/content/drive/MyDrive/Ketastasia/models/'
os.makedirs(models_dir, exist_ok=True)

drive_model_path = os.path.join(models_dir, 'pipeline2A_best_lstm.h5')

if 'model' in locals():
    model.save(drive_model_path)
    print(f" მოდელი წარმატებით შეინახა Google Drive-ზე: {drive_model_path}")
else:
    print(" ყურადღება: 'model' ცვლადი ვერ მოიძებნა მეხსიერებაში!")


run = wandb.init(
    project="ildolcefarniente",
    job_type="model_registration_lstm",
    name="register_pipeline2_lstm"
)

artifact = wandb.Artifact(
    name="pipeline2_lstm_models",
    type="model",
    description=f"Best 1-layer LSTM model with 64 hidden units and 0.4 dropout. Val Acc: {best_val_acc:.4f}, Val F1-Macro: {best_val_f1_macro:.4f}, Gap: {overfitting_gap:.4f}."
)


artifact.add_file(drive_model_path)


run.log_artifact(artifact, aliases=["latest"])
run.finish()

print("\n მორჩა, ყველაფერი სინქრონიზებულია და მზადაა!")

საუკეთესო შერჩეული მოდელი: lstm_dropout_0.4
Validation Accuracy: 0.6797
Validation F1-Macro: 0.5621
Overfitting Gap: 0.1425
 მოდელი წარმატებით შეინახა Google Drive-ზე: /content/drive/MyDrive/Ketastasia/models/pipeline2A_best_lstm.h5



 მორჩა, ყველაფერი სინქრონიზებულია და მზადაა!


In [ ]:
# =====================================================================
# Cell: საუკეთესო BiLSTM მოდელის ავტომატური შენახვა Drive-ზე და WandB-ზე
# =====================================================================
import os
import wandb

# 1. განვსაზღვროთ ჩვენი გამარჯვებული მოდელის მონაცემები
best_run_name = "bi_lstm_h128_b32"
best_val_f1_macro = 0.5957  # შენი BiLSTM-ის ზუსტი F1 ქულა
best_val_acc = 0.6812       # ვალიდაციის სიზუსტე
overfitting_gap = 0.1645    # train_acc - val_acc (დაახლოებითი)

print(f"საუკეთესო შერჩეული მოდელი: {best_run_name}")
print(f"Validation Accuracy: {best_val_acc:.4f}")
print(f"Validation F1-Macro: {best_val_f1_macro:.4f}")
print(f"Overfitting Gap: {overfitting_gap:.4f}")

# 2. მივუთითებთ საქაღალდეს Google Drive-ზე (უკვე არსებულს)
models_dir = '/content/drive/MyDrive/Ketastasia/models/'
drive_model_path = os.path.join(models_dir, 'pipeline2A_best_bilstm.h5')

if 'model' in locals():
    model.save(drive_model_path)
    print(f" მოდელი წარმატებით შეინახა Google Drive-ზე: {drive_model_path}")
else:
    print(" ყურადღება: 'model' ცვლადი ვერ მოიძებნა მეხსიერებაში!")


run = wandb.init(
    project="ildolcefarniente",
    job_type="model_registration_bilstm",
    name="register_pipeline2A_bilstm"
)

artifact = wandb.Artifact(
    name="pipeline2_bilstm_models",
    type="model",
    description=f"Best 1-layer Bidirectional LSTM model with 128 hidden units. Val Acc: {best_val_acc:.4f}, Val F1-Macro: {best_val_f1_macro:.4f}, Gap: {overfitting_gap:.4f}."
)


artifact.add_file(drive_model_path)


run.log_artifact(artifact, aliases=["latest"])
run.finish()

print("\n მორჩა, ორივე ტოპ მოდელი დაარქივებულია და მზადაა! 🚀🏆")

საუკეთესო შერჩეული მოდელი: bi_lstm_h128_b32
Validation Accuracy: 0.6812
Validation F1-Macro: 0.5957
Overfitting Gap: 0.1645
 მოდელი წარმატებით შეინახა Google Drive-ზე: /content/drive/MyDrive/Ketastasia/models/pipeline2A_best_bilstm.h5



 მორჩა, ორივე ტოპ მოდელი დაარქივებულია და მზადაა! 🚀🏆
